# S04.3 – Group leakage: один объект попал и в train, и в test

Цель: увидеть, как метрики улетают, если данные содержат повторяющиеся сущности (user/device/session) и мы делаем случайный split.

## Что вы освоите
- Понять, что такое групповой split и зачем он нужен
- Воспроизвести завышение метрик при случайном split
- Освоить GroupShuffleSplit / GroupKFold как анти-leakage инструмент

## Важные оговорки
- Этот кейс особенно важен для системной безопасности, где много данных по одним и тем же устройствам/пользователям.
- Мы используем синтетические данные, чтобы эффект был гарантирован и понятен.

---


## Среда, воспроизводимость и стиль эксперимента

Перед кодом – несколько правил, которые будут повторяться во всех ноутбуках:

1) **Воспроизводимость**: фиксируем `random_state` / seed.  
2) **Разделение данных**: test‑часть – это *священная зона*. Мы смотрим на неё только в конце.  
3) **Всё, что "обучается" (`.fit`)** должно видеть только train‑данные (иначе легко получить утечки).  
4) **Sanity‑checks**: после каждого шага проверяем, что получился ожидаемый результат (формы, распределения, пересечения и т.д.).


In [1]:
# Импорты: минимальный, но достаточный набор
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Для красивых картинок (простая визуализация)
import matplotlib.pyplot as plt

# Фиксируем seed для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.0.2
pandas: 2.2.2


## Общие функции (оценка моделей и печать метрик)

Чтобы не копировать одно и то же вручную, заведём пару функций.

Важно: эти функции *ничего не делают магически*. Мы специально пишем их максимально прозрачно,
чтобы вы видели, какие именно метрики считаются и на каких данных.


In [2]:
def summarize_binary_metrics(y_true, y_pred, *, positive_label=1):
    """Считает базовые метрики бинарной классификации.

    Мы считаем:
    - accuracy: доля верных ответов
    - precision: насколько "чистые" наши позитивные предсказания
    - recall: насколько хорошо мы находим позитивный класс
    - f1: гармоническое среднее precision и recall

    Почему это важно: в задачах безопасности цена FP и FN может быть разной.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
    }

def print_confusion(y_true, y_pred, labels=(0,1)):
    """Печать матрицы ошибок и пояснения, что есть что."""
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    display(df)
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn} (верно: 0), FP={fp} (ложная тревога), FN={fn} (пропуск), TP={tp} (верно: 1)")
    return cm

def evaluate_model(model, X_train, y_train, X_test, y_test, *, model_name="model"):
    """Обучает модель на train и оценивает на test. Возвращает словарь метрик."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = summarize_binary_metrics(y_test, y_pred)
    print(f"=== {model_name} ===")
    print(metrics)
    print("Confusion matrix:")
    _ = print_confusion(y_test, y_pred)
    return metrics


## Синтетические данные с 'сущностями' (groups)

Представим, что у нас есть:
- `user_id` или `device_id`,
- по каждому объекту много наблюдений,
- и у каждого объекта есть "подпись" (стабильное смещение), по которой модель может его узнавать.

Если случайно разделить строки на train/test,
часть строк одного и того же `user_id` окажется и там, и там.
Тогда модель фактически **узнаёт** пользователя, а не решает задачу – метрики завышаются.

Правильное решение: split по группам.


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(RANDOM_STATE)

n_groups = 500     # пользователей/устройств
rows_per_group = 10
n = n_groups * rows_per_group

group_id = np.repeat(np.arange(n_groups), rows_per_group)

# Стабильная "подпись" группы (например, особенности устройства)
group_signature = rng.normal(0, 1.0, size=n_groups)
sig = group_signature[group_id]

# Признаки: один сильный признак связан с подписью, остальные шум
X = np.column_stack([
    sig + rng.normal(0, 0.3, size=n),              # f0 несёт подпись группы
    rng.normal(0, 1.0, size=n),
    rng.normal(0, 1.0, size=n),
    rng.normal(0, 1.0, size=n),
])

# Метка y зависит от подписи (чтобы была задача)
y = (sig + rng.normal(0, 0.5, size=n) > 0).astype(int)

X = pd.DataFrame(X, columns=["f0_signature_like", "f1", "f2", "f3"])
y = pd.Series(y, name="y")
groups = pd.Series(group_id, name="group_id")

print("Data shape:", X.shape)
print("Class balance:", y.value_counts(normalize=True).to_dict())


Data shape: (5000, 4)
Class balance: {0: 0.5042, 1: 0.4958}


## Плохой протокол: случайный split (группы пересекаются)

Мы делаем обычный `train_test_split`. Он не знает про `group_id`,
поэтому один и тот же `group_id` почти наверняка окажется и в train, и в test.


In [4]:
X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, groups,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

# Проверим пересечение групп
intersect = set(g_train.unique()).intersection(set(g_test.unique()))
print("Количество пересекающихся групп:", len(intersect), "из", n_groups)
print("Доля пересечений:", len(intersect)/n_groups)

model = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

bad = summarize_binary_metrics(y_test, y_pred)
print("BAD (random split) metrics:", bad)


Количество пересекающихся групп: 477 из 500
Доля пересечений: 0.954
BAD (random split) metrics: {'accuracy': 0.8096, 'precision': 0.805111821086262, 'recall': 0.8129032258064516, 'f1': 0.8089887640449438}


## Хороший протокол: split по группам (GroupShuffleSplit)

Теперь мы гарантируем: каждый `group_id` находится либо в train, либо в test.


In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train2, X_test2 = X.iloc[train_idx], X.iloc[test_idx]
y_train2, y_test2 = y.iloc[train_idx], y.iloc[test_idx]
g_train2, g_test2 = groups.iloc[train_idx], groups.iloc[test_idx]

intersect2 = set(g_train2.unique()).intersection(set(g_test2.unique()))
print("Пересечение групп (должно быть 0):", len(intersect2))

model2 = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)

good = summarize_binary_metrics(y_test2, y_pred2)
print("GOOD (group split) metrics:", good)


Пересечение групп (должно быть 0): 0
GOOD (group split) metrics: {'accuracy': 0.8352, 'precision': 0.8202614379084967, 'recall': 0.8394648829431438, 'f1': 0.8297520661157025}


## Сравнение и главный вывод

Если метрики в плохом протоколе существенно выше, это значит:
- модель частично "узнала" группы,
- оценка качества завышена,
- в реальном использовании на новых группах качество будет хуже.

Правило:
> Если в данных есть сущности (пользователь/устройство/сессия/документ), split должен учитывать их.

Это один из самых частых и самых болезненных источников утечек.


In [6]:
summary = pd.DataFrame([
    {"protocol": "BAD random split", **bad},
    {"protocol": "GOOD group split", **good},
]).set_index("protocol")
display(summary)


,accuracy,precision,recall,f1
protocol,,,,
BAD random split,0.8096,0.805112,0.812903,0.808989
GOOD group split,0.8352,0.820261,0.839465,0.829752
